# Cleaned analysis notebook

This copy preserves the original analysis/calculation cells. Personal machine paths have been replaced with project-relative paths.
Place input workbooks and data in `data/`; generated files are written to `outputs/`. Create these folders before running.
Saved cell outputs were cleared so results are regenerated when the notebook is run.


In [ ]:
from pathlib import Path
import os

# Set this to the local folder containing nnUNet_raw, nnUNet_preprocessed, and nnUNet_results.
BASE_DIR = Path("nnunet_workspace")
DATASET_ID = "Dataset002_SPIDER"
n = 0  # Set to the desired n index, e.g. 0, 1, 2, 3, or 4

RAW_DIR = BASE_DIR / "nnUNet_raw" / DATASET_ID
PREPROCESSED_DIR = BASE_DIR / "nnUNet_preprocessed" / DATASET_ID
RESULTS_DIR = BASE_DIR / "nnUNet_results" / DATASET_ID

GROUND_TRUTH_DIR = RAW_DIR / f"groundtruth_fold{n}"
PREDICTIONS_DIR = RESULTS_DIR / f"predictions_fold{n}"
IMAGES_TS_DIR = GROUND_TRUTH_DIR / "imagesTs"
LABELS_TS_DIR = GROUND_TRUTH_DIR / "labelsTs"

# nnU-Net v2 environment variables
os.environ["nnUNet_raw"] = str(BASE_DIR / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(BASE_DIR / "nnUNet_preprocessed")
os.environ["nnUNet_results"] = str(BASE_DIR / "nnUNet_results")

In [ ]:
import shutil

imagesTr = RAW_DIR / "imagesTr"
labelsTr = RAW_DIR / "labelsTr"

IMAGES_TS_DIR.mkdir(parents=True, exist_ok=True)
LABELS_TS_DIR.mkdir(parents=True, exist_ok=True)

for case in val_cases:
    shutil.copy2(imagesTr / f"{case}_0000.nii.gz", IMAGES_TS_DIR / f"{case}_0000.nii.gz")
    shutil.copy2(labelsTr / f"{case}.nii.gz", LABELS_TS_DIR / f"{case}.nii.gz")

In [ ]:
import os
import subprocess

env = os.environ.copy()
env["nnUNet_n_proc_DA"] = "0"
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"

command = [
    "nnUNetv2_predict",
    "-i", str(IMAGES_TS_DIR),
    "-o", str(PREDICTIONS_DIR),
    "-d", "2",
    "-c", "3d_fullres",
    "-f", str(n),
    "-chk", "checkpoint_best.pth",
]

subprocess.run(command, check=True, env=env)

In [ ]:
import os
import numpy as np
import nibabel as nib

gt_dir = str(LABELS_TS_DIR)
pred_dir = str(PREDICTIONS_DIR)

def dice_coefficient(gt, pred):
    gt = gt > 0
    pred = pred > 0
    intersection = np.logical_and(gt, pred).sum()
    return (2.0 * intersection) / (gt.sum() + pred.sum() + 1e-8)

dice_scores = []

for fname in sorted(os.listdir(gt_dir)):
    if not fname.endswith(".nii.gz"):
        continue

    gt = nib.load(os.path.join(gt_dir, fname)).get_fdata()
    pred = nib.load(os.path.join(pred_dir, fname)).get_fdata()

    d = dice_coefficient(gt, pred)
    dice_scores.append(d)

    print(f"{fname}: Dice = {d:.4f}")

print("\n✅ Mean Dice:", np.mean(dice_scores))
print("✅ Std Dice:", np.std(dice_scores))

In [ ]:
import SimpleITK as sitk
import numpy as np
from medpy.metric import binary

import os

gt_dir = str(LABELS_TS_DIR)
pred_dir = str(PREDICTIONS_DIR)

labels = [1,2,3,4,5,6,7,8,9,10,11]  # change if multi-class (e.g., [1,2,3])

results = {}

for case in sorted(os.listdir(gt_dir)):
    if not case.endswith(".nii.gz"):
        continue

    gt = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(gt_dir, case)))
    pred = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(pred_dir, case)))

    results[case] = {}

    for lbl in labels:
        gt_bin = (gt == lbl)
        pred_bin = (pred == lbl)

        if gt_bin.sum() == 0:
            continue

        dice = binary.dc(pred_bin, gt_bin)
        hd95 = binary.hd95(pred_bin, gt_bin)
        asd = binary.asd(pred_bin, gt_bin)

        results[case][f"class_{lbl}"] = {
            "Dice": dice,
            "HD95": hd95,
            "ASD": asd
        }

results

In [ ]:
#HD95 and ASD for vertebrae and IVDs seperately
import numpy as np
from medpy.metric import binary
# Define label IDs
VERTEBRA_LABELS = [1, 2, 3, 4, 5]          # adjust if needed
IVD_LABELS      = [6, 7, 8, 9, 10, 11]     # adjust if needed
# Vertebra masks
gt_vert   = np.isin(gt, VERTEBRA_LABELS)
pred_vert = np.isin(pred, VERTEBRA_LABELS)

# IVD masks
gt_ivd   = np.isin(gt, IVD_LABELS)
pred_ivd = np.isin(pred, IVD_LABELS)

#Compute HD95 and ASD per structure
#Vertebrae
if gt_vert.any() and pred_vert.any():
    dice_vert = binary.dc(pred_vert, gt_vert)
    hd95_vert = binary.hd95(pred_vert, gt_vert)
    asd_vert  = binary.asd(pred_vert, gt_vert)
else:
    dice_vert = np.nan
    hd95_vert = np.nan
    asd_vert  = np.nan

print("Vertebrae Dice:", dice_vert)
print("Vertebrae HD95:", hd95_vert)
print("Vertebrae ASD :", asd_vert)

#IVD
if gt_ivd.any() and pred_ivd.any():
    dice_ivd = binary.dc(pred_ivd, gt_ivd)
    hd95_ivd = binary.hd95(pred_ivd, gt_ivd)
    asd_ivd  = binary.asd(pred_ivd, gt_ivd)
else:
    dice_ivd = np.nan
    hd95_ivd = np.nan
    asd_ivd  = np.nan

print("IVD Dice:", dice_ivd)
print("IVD HD95:", hd95_ivd)
print("IVD ASD :", asd_ivd)

In [ ]:
import numpy as np
import pandas as pd

num_classes = 11
table_rows = []

for idx, (case, class_dict) in enumerate(sorted(results.items()), start=1):
    row = {
        "Number": idx,
        "Case": case
    }

    # Initialize all class columns with NaN
    for c in range(1, num_classes + 1):
        row[f"Class_{c}"] = np.nan

    # Fill Dice values
    for cls_name, metrics in class_dict.items():
        cls_id = int(cls_name.replace("class_", ""))
        row[f"Class_{cls_id}"] = metrics["Dice"]

    table_rows.append(row)

df = pd.DataFrame(table_rows)
df

In [ ]:
import numpy as np

class_cols = [f"Class_{i}" for i in range(1, 12)]

mean_dice_per_class = df[class_cols].mean()
overall_mean_dice = mean_dice_per_class.mean()
overall_std_dice = mean_dice_per_class.std()

print(mean_dice_per_class)
print(f"\nOverall Mean Dice = {overall_mean_dice:.3f} ± {overall_std_dice:.3f}")

In [ ]:
# Create / assign DataFrame
df_file = df.copy()

# Save to CSV
df_file.to_csv(f"dice_fold{n}.csv", index=False)

# **Wilcoxon signed-rank test**

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

# --------------------------------------------------
# 1. Load the per-patient Dice data
# --------------------------------------------------

file_path ="data/Original_dice_all_fold.xlsx"

# Use pd.read_excel for .xlsx files
df = pd.read_excel(file_path)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

In [ ]:
# --------------------------------------------------
# 2. Define vertebra and IVD classes
# --------------------------------------------------

vertebra_cols = [
    "Class_1",
    "Class_2",
    "Class_3",
    "Class_4",
    "Class_5"
]

ivd_cols = [
    "Class_6",
    "Class_7",
    "Class_8",
    "Class_9",
    "Class_10",
    "Class_11"
]

# Check that all columns exist
print("Missing vertebra columns:",
      [c for c in vertebra_cols if c not in df.columns])

print("Missing IVD columns:",
      [c for c in ivd_cols if c not in df.columns])

Calculate patient-level mean vertebral and IVD Dice

In [ ]:
# --------------------------------------------------
# 3. Patient-level mean Dice
# --------------------------------------------------

df["mean_vertebra_dice"] = df[vertebra_cols].mean(axis=1, skipna=True)

df["mean_ivd_dice"] = df[ivd_cols].mean(axis=1, skipna=True)

print(
    df[
        ["mean_vertebra_dice", "mean_ivd_dice"]
    ].head()
)

In [ ]:
# --------------------------------------------------
# 4. Keep patients with both measurements
# --------------------------------------------------

paired = df[
    ["mean_vertebra_dice", "mean_ivd_dice"]
].dropna()

print("Number of paired patients:", len(paired))

Wilcoxon signed-rank test

In [ ]:
# --------------------------------------------------
# 5. Wilcoxon signed-rank test
# --------------------------------------------------

stat, p_value = wilcoxon(
    paired["mean_vertebra_dice"],
    paired["mean_ivd_dice"],
    alternative="two-sided"
)

print("Wilcoxon signed-rank statistic:", stat)
print("p-value:", p_value)

patient-level descriptive values

In [ ]:
# --------------------------------------------------
# 6. Descriptive statistics for patient-level means
# --------------------------------------------------

print("\nPatient-level vertebral Dice:")
print("Mean:", paired["mean_vertebra_dice"].mean())
print("SD:", paired["mean_vertebra_dice"].std())
print("Median:", paired["mean_vertebra_dice"].median())

print("\nPatient-level IVD Dice:")
print("Mean:", paired["mean_ivd_dice"].mean())
print("SD:", paired["mean_ivd_dice"].std())
print("Median:", paired["mean_ivd_dice"].median())

paired difference

In [ ]:
# --------------------------------------------------
# 7. Paired difference
# --------------------------------------------------

paired["difference"] = (
    paired["mean_vertebra_dice"]
    - paired["mean_ivd_dice"]
)

print("\nMean paired difference:",
      paired["difference"].mean())

print("Median paired difference:",
      paired["difference"].median())

print("SD paired difference:",
      paired["difference"].std())

calculate the effect size

In [ ]:
import numpy as np

x = paired["mean_vertebra_dice"].to_numpy()
y = paired["mean_ivd_dice"].to_numpy()

differences = x - y

# Remove zero differences
differences_nonzero = differences[differences != 0]

n = len(differences_nonzero)

# Absolute differences and ranks
abs_diff = np.abs(differences_nonzero)

ranks = pd.Series(abs_diff).rank(method="average").to_numpy()

W_plus = ranks[differences_nonzero > 0].sum()
W_minus = ranks[differences_nonzero < 0].sum()

rank_biserial = (W_plus - W_minus) / (W_plus + W_minus)

print("Rank-biserial correlation:", rank_biserial)

In [ ]:
df["n_vertebra"] = df[vertebra_cols].notna().sum(axis=1)
df["n_ivd"] = df[ivd_cols].notna().sum(axis=1)

print(df["n_vertebra"].value_counts().sort_index())
print(df["n_ivd"].value_counts().sort_index())

In [ ]:
print(
    df[
        ["n_vertebra", "n_ivd"]
    ].describe()
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

# --------------------------------------------------
# 3. Keep paired observations
# --------------------------------------------------

paired = df[
    ["mean_vertebra_dice", "mean_ivd_dice"]
].dropna().copy()

print("Number of paired patients:", len(paired))

# --------------------------------------------------
# 4. Wilcoxon signed-rank test
# --------------------------------------------------

stat, p_value = wilcoxon(
    paired["mean_vertebra_dice"],
    paired["mean_ivd_dice"],
    alternative="two-sided"
)

print("Wilcoxon statistic:", stat)
print("p-value:", p_value)

# --------------------------------------------------
# 5. Prepare plotting data
# --------------------------------------------------

vertebra = paired["mean_vertebra_dice"].to_numpy()
ivd = paired["mean_ivd_dice"].to_numpy()

data = [vertebra, ivd]

# --------------------------------------------------
# 6. Create figure
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 6))

# Boxplots
bp = ax.boxplot(
    data,
    positions=[1, 2],
    widths=0.45,
    patch_artist=False,
    showfliers=False
)

# --------------------------------------------------
# 7. Plot individual patient values
# --------------------------------------------------

rng = np.random.default_rng(42)

x1 = 1 + rng.uniform(-0.08, 0.08, size=len(vertebra))
x2 = 2 + rng.uniform(-0.08, 0.08, size=len(ivd))

ax.scatter(
    x1,
    vertebra,
    s=18,
    alpha=0.45
)

ax.scatter(
    x2,
    ivd,
    s=18,
    alpha=0.45
)

# --------------------------------------------------
# 8. Connect paired observations
# --------------------------------------------------

for i in range(len(paired)):
    ax.plot(
        [x1[i], x2[i]],
        [vertebra[i], ivd[i]],
        linewidth=0.5,
        alpha=0.15
    )

# --------------------------------------------------
# 9. Labels
# --------------------------------------------------

ax.set_xticks([1, 2])
ax.set_xticklabels([
    "Vertebrae",
    "IVDs"
])

ax.set_ylabel("Patient-level mean Dice coefficient")

ax.set_title(
    "Patient-level Dice comparison between vertebrae and IVDs"
)

# --------------------------------------------------
# 10. Add Wilcoxon p-value
# --------------------------------------------------

y_max = max(
    np.max(vertebra),
    np.max(ivd)
)

y_line = y_max + 0.025

ax.plot(
    [1, 1, 2, 2],
    [y_line - 0.005, y_line, y_line, y_line - 0.005],
    linewidth=1
)

if p_value < 0.001:
    p_text = "Wilcoxon p < 0.001"
else:
    p_text = f"Wilcoxon p = {p_value:.3g}"

ax.text(
    1.5,
    y_line + 0.005,
    p_text,
    ha="center",
    va="bottom"
)

# --------------------------------------------------
# 11. Formatting
# --------------------------------------------------

ax.set_ylim(
    min(
        np.min(vertebra),
        np.min(ivd)
    ) - 0.03,
    y_line + 0.06
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# --------------------------------------------------
# 12. Save
# --------------------------------------------------

plt.savefig(
    "outputs/patient_level_vertebra_vs_ivd_wilcoxon.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "outputs/patient_level_vertebra_vs_ivd_wilcoxon.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()